In [ ]:
# 1. Install and Import
!pip install gradio -q
import sqlite3
import pandas as pd
import gradio as gr
import os
import matplotlib.pyplot as plt

DB_NAME = "mall_membership_pro.db"
CSV_URL = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"

# --- Database Setup Logic ---
def init_db():
    if os.path.exists(DB_NAME):
        os.remove(DB_NAME)

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS customers (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE,
                full_name TEXT,
                age INTEGER,
                city TEXT,
                tier TEXT
            )
        """)

        try:
            df = pd.read_csv(CSV_URL).head(10)
            data = df.apply(lambda row: (
                row['고객ID'], f"Member {row['고객ID']}", int(row['연령']), row['지역'], "Standard"
            ), axis=1).tolist()
            cursor.executemany("INSERT INTO customers (username, full_name, age, city, tier) VALUES (?, ?, ?, ?, ?)", data)
            conn.commit()
        except Exception as e:
            print(f"Initialization Error: {e}")

# --- Core CRUD Functions ---
def view_data():
    with sqlite3.connect(DB_NAME) as conn:
        df = pd.read_sql_query("SELECT * FROM customers", conn)
        df.columns = ["Index", "Membership ID", "Member Name", "Age", "Location", "Membership Tier"]
    return df

def add_user(username, full_name, age, city, tier):
    if not username: return "System Alert: Membership ID Required", view_data()
    try:
        with sqlite3.connect(DB_NAME) as conn:
            cursor = conn.cursor()
            cursor.execute("INSERT INTO customers (username, full_name, age, city, tier) VALUES (?, ?, ?, ?, ?)", (username, full_name, age, city, tier))
            conn.commit()
        return f"Success: Created Record {username}", view_data()
    except Exception as e:
        return f"Error: {e}", view_data()

def update_user(username, full_name, age, city, tier):
    try:
        with sqlite3.connect(DB_NAME) as conn:
            cursor = conn.cursor()
            cursor.execute("UPDATE customers SET full_name=?, age=?, city=?, tier=? WHERE username=?", (full_name, age, city, tier, username))
            conn.commit()
        return f"Success: Updated Member {username}", view_data()
    except Exception as e:
        return f"Error: {e}", view_data()

def delete_user(username):
    try:
        with sqlite3.connect(DB_NAME) as conn:
            cursor = conn.cursor()
            cursor.execute("DELETE FROM customers WHERE username = ?", (username,))
            conn.commit()
        return f"Success: Terminated Membership {username}", view_data()
    except Exception as e:
        return f"Error: {e}", view_data()

# --- INNOVATION 1: Live Analytics Dashboard ---
def generate_dashboard():
    with sqlite3.connect(DB_NAME) as conn:
        df = pd.read_sql_query("SELECT * FROM customers", conn)

    if df.empty:
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "No Data Available", ha='center', va='center')
        return fig

    # Create a figure with 2 subplots (1 row, 2 columns)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # 1. Tier Distribution (Pie Chart)
    tier_counts = df['tier'].value_counts()
    ax1.pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=140,
            colors=['#1a2a6c', '#b21f1f', '#fdbb2d', '#2c3e50'])
    ax1.set_title('Membership Tier Distribution', fontweight='bold')

    # 2. Age Demographics (Histogram)
    ax2.hist(df['age'], bins=10, color='#1a2a6c', edgecolor='white')
    ax2.set_title('Age Demographics', fontweight='bold')
    ax2.set_xlabel('Age')
    ax2.set_ylabel('Number of Members')

    plt.tight_layout()
    return fig

# --- INNOVATION 2: Smart Marketing Assistant ---
def generate_marketing_message(username):
    if not username: return "Please enter a valid Membership ID."
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT full_name, age, city, tier FROM customers WHERE username=?", (username,))
        result = cursor.fetchone()

    if not result:
        return "Customer not found in database."

    name, age, city, tier = result

    # Dynamic message generation based on tier and location
    discount = "10%" if tier == "Standard" else "20%" if tier == "Silver" else "30%" if tier == "Gold" else "50% VIP"
    perk = "Free Parking" if tier in ["Gold", "VIP"] else "Double Points"

    msg = f"📩 EMAIL DRAFT GENERATED:\n"
    msg += f"-----------------------------------\n"
    msg += f"Subject: Exclusive {tier} Offer for {name}!\n\n"
    msg += f"Dear {name},\n\n"
    msg += f"Thank you for being a valued {tier} member at our {city} branch! "
    msg += f"Because we appreciate our loyal shoppers, we are giving you an exclusive {discount} discount "
    msg += f"on your next purchase this weekend. Plus, enjoy {perk} on us!\n\n"
    msg += f"We look forward to seeing you soon.\n"
    msg += f"- The Retail Management Team"

    return msg

# --- Professional Styling ---
custom_css = """
.gradio-container { font-family: 'Helvetica', 'Arial', sans-serif !important; }
.main-header { text-align: center; color: #1a2a6c; border-bottom: 2px solid #1a2a6c; margin-bottom: 20px; padding-bottom: 10px; }
button { border-radius: 4px !important; text-transform: uppercase !important; font-weight: bold !important; }
"""

# --- Build Interface ---
init_db()

with gr.Blocks(theme=gr.themes.Default(primary_hue="slate", radius_size="none"), css=custom_css) as demo:
    gr.Markdown("# 🏢 RETAIL CRM & MEMBERSHIP SYSTEM", elem_classes="main-header")

    with gr.Tabs():

        # TAB 1: Database Operations (Original)
        with gr.Tab("📋 Membership Records"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### DATA ENTRY")
                    u_name = gr.Textbox(label="MEMBERSHIP ID", placeholder="CUST_XXXX")
                    f_name = gr.Textbox(label="LEGAL NAME")

                    with gr.Row():
                        age_input = gr.Number(label="AGE", value=25)
                        city_input = gr.Textbox(label="LOCATION")

                    tier_input = gr.Dropdown(label="MEMBERSHIP TIER", choices=["Standard", "Silver", "Gold", "VIP"], value="Standard")

                    gr.Markdown("### OPERATIONS")
                    with gr.Row():
                        btn_add = gr.Button("➕ ADD", variant="primary")
                        btn_update = gr.Button("✏️ UPDATE")
                        btn_delete = gr.Button("🗑️ DELETE", variant="stop")

                    status_msg = gr.Textbox(label="SYSTEM STATUS", interactive=False)

                with gr.Column(scale=2):
                    gr.Markdown("### MASTER DATABASE")
                    db_display = gr.Dataframe(value=view_data(), interactive=False)
                    btn_refresh = gr.Button("🔄 REFRESH DATA")

            # Wire up Tab 1
            btn_add.click(add_user, [u_name, f_name, age_input, city_input, tier_input], [status_msg, db_display])
            btn_update.click(update_user, [u_name, f_name, age_input, city_input, tier_input], [status_msg, db_display])
            btn_delete.click(delete_user, [u_name], [status_msg, db_display])
            btn_refresh.click(view_data, None, db_display)

        # TAB 2: Live Analytics (New!)
        with gr.Tab("📊 Retail Analytics"):
            gr.Markdown("### REAL-TIME DEMOGRAPHICS DASHBOARD")
            gr.Markdown("View your customer breakdown instantly. Updates automatically when you refresh.")
            plot_output = gr.Plot(value=generate_dashboard())
            btn_refresh_plot = gr.Button("🔄 REFRESH CHARTS", variant="primary")

            btn_refresh_plot.click(generate_dashboard, None, plot_output)

        # TAB 3: Smart Marketing (New!)
        with gr.Tab("💡 Smart Marketing Assistant"):
            gr.Markdown("### AUTOMATED PROMOTIONS")
            gr.Markdown("Type a Membership ID to automatically generate a personalized marketing email based on their tier and city.")

            with gr.Row():
                with gr.Column():
                    target_customer = gr.Textbox(label="Target Membership ID", placeholder="e.g., CUST_0001")
                    btn_generate = gr.Button("✨ GENERATE CAMPAIGN", variant="primary")
                with gr.Column():
                    marketing_output = gr.Textbox(label="Generated Marketing Material", lines=10, interactive=False)

            btn_generate.click(generate_marketing_message, inputs=[target_customer], outputs=[marketing_output])

# Launch the app
demo.launch(debug=True)

Initialization Error: '연령'


/tmp/ipykernel_2891/739116230.py:145: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue="slate", radius_size="none"), css=custom_css) as demo:
/tmp/ipykernel_2891/739116230.py:145: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue="slate", radius_size="none"), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b0f02c453a6efe3d02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
